In [50]:
import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import os

In [51]:
cohort = "brca"

data_path = '../../data/clinical_data'
split_folder = 'shap_experiment'

filename = f'{cohort}_clinical'
n_folds = 5

In [52]:
# loading data

df = pd.read_csv(f"{data_path}/{filename}.csv", index_col=0)
df.head()

,case_id,dss_survival_days,dss_censorship
0,TCGA-3C-AAAU,4047.0,1
1,TCGA-3C-AALI,4005.0,1
2,TCGA-3C-AALJ,1474.0,1
3,TCGA-3C-AALK,1448.0,1
4,TCGA-4H-AAAK,348.0,1


In [53]:
# identifying "location" from patient_id
df['location'] = df['case_id'].apply(lambda x: x.split('-')[1])

In [54]:
# tylko prawdziwe zdarzenia
events_df = df[df['dss_censorship'] == 0]

# maksymalny czas zdarzenia
max_event_time = events_df['dss_survival_days'].max()

# wszystkie obserwacje z tym czasem
critical_idx = events_df[
    events_df['dss_survival_days'] == max_event_time
].index


In [55]:
df_rest = df.drop(index=critical_idx).reset_index(drop=True)
critical_df = df.loc[critical_idx].reset_index(drop=True)


In [56]:
n_outer = 5
n_inner_folds = 5

split_dir = f"{data_path}/{filename}/{split_folder}"
os.makedirs(split_dir, exist_ok=True)

gss_outer = GroupKFold(n_splits=n_outer)

for outer_idx, (trainval_idx, holdout_idx) in enumerate(
    gss_outer.split(df_rest, groups=df_rest['location'])
):
    outer_path = os.path.join(split_dir, f"outer_{outer_idx}")
    os.makedirs(outer_path, exist_ok=True)

    # 🔥 NOWY folder na foldy
    splits_path = os.path.join(outer_path, "splits")
    os.makedirs(splits_path, exist_ok=True)

    trainval_df = df_rest.iloc[trainval_idx].reset_index(drop=True)
    holdout_df = df_rest.iloc[holdout_idx].reset_index(drop=True)

    # zapis globalnego testu
    holdout_df.to_csv(os.path.join(outer_path, "holdout.csv"), index=False)

    # zapis globalnego train
    trainval_df.to_csv(os.path.join(outer_path, "trainval.csv"), index=False)

    # === INNER CV ===
    gkf = GroupKFold(n_splits=n_inner_folds)

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(trainval_df, groups=trainval_df['location'])
    ):
        # 🔥 zmiana tutaj — foldy trafiają do /splits/
        fold_path = os.path.join(splits_path, f"{fold}")
        os.makedirs(fold_path, exist_ok=True)

        train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
        test_df = trainval_df.iloc[test_idx].reset_index(drop=True)

        # 🔥 survival leakage fix
        train_events = train_df[train_df['dss_censorship'] == 0]
        test_events = test_df[test_df['dss_censorship'] == 0]

        if len(test_events) > 0:
            max_train_event_time = train_events['dss_survival_days'].max()
            problematic = test_events[
                test_events['dss_survival_days'] > max_train_event_time
            ]

            if len(problematic) > 0:
                train_df = pd.concat([train_df, problematic], ignore_index=True)
                test_df = test_df.drop(problematic.index).reset_index(drop=True)

        # zapis
        train_df.to_csv(os.path.join(fold_path, "train_filtered.csv"), index=False)
        test_df.to_csv(os.path.join(fold_path, "test_filtered.csv"), index=False)

        print(
            f"[Outer {outer_idx} | Fold {fold}] "
            f"train={len(train_df)}, test={len(test_df)}, holdout={len(holdout_df)}"
        )

[Outer 0 | Fold 0] train=688, test=172, holdout=216
[Outer 0 | Fold 1] train=688, test=172, holdout=216
[Outer 0 | Fold 2] train=688, test=172, holdout=216
[Outer 0 | Fold 3] train=691, test=169, holdout=216
[Outer 0 | Fold 4] train=688, test=172, holdout=216
[Outer 1 | Fold 0] train=689, test=172, holdout=215
[Outer 1 | Fold 1] train=689, test=172, holdout=215
[Outer 1 | Fold 2] train=688, test=173, holdout=215
[Outer 1 | Fold 3] train=691, test=170, holdout=215
[Outer 1 | Fold 4] train=689, test=172, holdout=215
[Outer 2 | Fold 0] train=690, test=171, holdout=215
[Outer 2 | Fold 1] train=689, test=172, holdout=215
[Outer 2 | Fold 2] train=689, test=172, holdout=215
[Outer 2 | Fold 3] train=689, test=172, holdout=215
[Outer 2 | Fold 4] train=689, test=172, holdout=215
[Outer 3 | Fold 0] train=688, test=173, holdout=215
[Outer 3 | Fold 1] train=689, test=172, holdout=215
[Outer 3 | Fold 2] train=689, test=172, holdout=215
[Outer 3 | Fold 3] train=689, test=172, holdout=215
[Outer 3 | F